### Imports and installations

In [2]:
!pip install -q h3 duckdb pandas ray[data]
import ray
import duckdb
import pandas as pd
import h3
import glob
my_env = {
    "pip": ["h3", "duckdb", "pandas<3.0.0", "pyarrow"]
}

MY_NAMESPACE = "OBIS_TEST"


### Ray Resources

In [3]:
if ray.is_initialized():
    ray.shutdown()

ray.init(address="auto", runtime_env=my_env, namespace=MY_NAMESPACE)
print("Connected to Ray!")

ctx = ray.data.DataContext.get_current()
ctx.enable_rich_progress_bars = True
ctx.use_ray_tqdm = False
ctx.log_internal_stack_trace_to_stdout = True

2026-03-27 08:05:30,915	INFO worker.py:1669 -- Using address ray://10.10.1.98:10001 set in the environment variable RAY_ADDRESS
2026-03-27 08:05:30,945	INFO client_builder.py:241 -- Passing the following kwargs to ray.init() on the server: log_to_driver
SIGTERM handler is not set because current thread is not the main thread.


Connected to Ray!


# OBS Mapper

In [25]:
@ray.remote
class OBISProductionWorker:
    def process_all(self, input_pattern: str, output_path: str):
        import duckdb
        import h3
        import ray.data
        import glob
        import pandas as pd
        
        # 1. Grab every Parquet file across all species folders
        all_files = glob.glob(input_pattern, recursive=True)
        if len(all_files) == 0:
            return "Error: No files found matching the pattern."
            
        print(f"Found {len(all_files)} Parquet files. Building pipeline...")
        
        def process_obis_batch(batch: pd.DataFrame) -> pd.DataFrame:
            try:
                required_cols = [
                    'decimalLatitude', 'decimalLongitude', 'eventDate',
                    'date_year', 'month', 'day', 'date_start', 'date_end',
                    'scientificName', 'class', 'individualCount'
                ]
                for col in required_cols:
                    if col not in batch.columns:
                        batch[col] = pd.NA

                con = duckdb.connect()
                query = """
                    SELECT 
                        decimalLatitude, 
                        decimalLongitude, 
                        COALESCE(CAST(eventDate AS VARCHAR), CAST(date_year AS VARCHAR), CAST(date_start AS VARCHAR)) AS best_time_marker,
                        scientificName, 
                        class, 
                        individualCount
                    FROM batch
                    WHERE decimalLatitude IS NOT NULL 
                      AND decimalLongitude IS NOT NULL
                      AND scientificName IS NOT NULL
                """
                clean_df = con.execute(query).df()
                
                if not clean_df.empty:
                    clean_df['h3_index'] = clean_df.apply(
                        lambda row: h3.latlng_to_cell(row['decimalLatitude'], row['decimalLongitude'], 7), 
                        axis=1
                    )
                else:
                    clean_df['h3_index'] = pd.Series(dtype='str')
                    
                return clean_df
            
            except Exception as e:
                print(f"WORKER CRASHED ON DATA: {e}")
                raise e 

        # 2. Read all files
        ds = ray.data.read_parquet(all_files)
        
        # 3. Apply the processing
        processed_ds = ds.map_batches(process_obis_batch, batch_format="pandas")
        
        # 4. THE TRIGGER: Write the clean data back to disk natively
        print(f"Executing pipeline and writing to: {output_path}")
        processed_ds.write_parquet(output_path)
        
        return f"Success! All {len(all_files)} files processed and saved to {output_path}"

### Send to ray

In [ ]:
input_pattern = "/mnt/shared_data/finflow/obis_raw/**/*.parquet"

# Create a new folder for the clean, unified data
output_path = "/mnt/shared_data/finflow/obis_clean/"

print("Spinning up Production Worker...")
processor = OBISProductionWorker.options(name="OBIS_Prod_Worker", get_if_exists=True).remote()

print("Submitting massive OBIS processing job to cluster...")
# This will block until the entire dataset is processed and written
result = ray.get(processor.process_all.remote(input_pattern, output_path))

print(result)

In [6]:
import duckdb

print("Counting total cleaned rows...")

query = """
    SELECT count(*) AS total_clean_rows 
    FROM read_parquet('/mnt/shared_data/finflow/gfw_clean/*.parquet')
"""

total_df = duckdb.query(query).df()
display(total_df)

Counting total cleaned rows...


,total_clean_rows
0,647519606


# GFW Mapper

In [4]:
@ray.remote
class GFWProductionWorker:
    def process_all(self, input_pattern: str, output_path: str):
        import duckdb
        import h3
        import ray.data
        import glob
        import pandas as pd
        import re
        
        all_files = glob.glob(input_pattern, recursive=True)
        if len(all_files) == 0:
            return "Error: No files found matching the pattern."
            
        print(f"Found {len(all_files)} GFW Parquet files. Building pipeline...")
        
        def process_gfw_batch(batch: pd.DataFrame) -> pd.DataFrame:
            try:
                # 1. EXTRACT TIME FROM FILENAME
                # Since we used include_paths=True, 'path' is a column.
                # We pull out the "YYYY-MM" format from the file path.
                if 'path' in batch.columns:
                    batch['time_marker'] = batch['path'].str.extract(r'(\d{4}-\d{2})')[0]
                else:
                    batch['time_marker'] = pd.NA

                # 2. RUN SQL QUERY
                # Notice how clean this is compared to OBIS!
                con = duckdb.connect()
                query = """
                    SELECT 
                        lat AS decimalLatitude, 
                        lon AS decimalLongitude, 
                        time_marker AS best_time_marker,
                        hours AS fishing_hours
                    FROM batch
                    WHERE lat IS NOT NULL 
                      AND lon IS NOT NULL
                """
                clean_df = con.execute(query).df()
                
                # 3. APPLY H3 INDEX
                if not clean_df.empty:
                    clean_df['h3_index'] = clean_df.apply(
                        lambda row: h3.latlng_to_cell(row['decimalLatitude'], row['decimalLongitude'], 7), 
                        axis=1
                    )
                else:
                    clean_df['h3_index'] = pd.Series(dtype='str')
                    
                # Drop the original coordinates if you strictly want to save space, 
                # but we'll keep them for consistency with the OBIS output for now.
                return clean_df
            
            except Exception as e:
                print(f"WORKER CRASHED ON DATA: {e}")
                raise e 

        # 4. READ FILES WITH PATHS INCLUDED
        # include_paths=True is the secret that saves our time dimension!
        ds = ray.data.read_parquet(all_files, include_paths=True)
        
        # 5. EXECUTE AND WRITE
        processed_ds = ds.map_batches(process_gfw_batch, batch_format="pandas")
        
        print(f"Executing pipeline and writing to: {output_path}")
        processed_ds.write_parquet(output_path)
        
        return f"Success! All {len(all_files)} GFW files processed and saved to {output_path}"

### Send to Ray

#### Set up new detached Actor

In [ ]:
input_pattern = "/mnt/shared_data/finflow/gfw_raw/**/*.parquet"

output_path = "/mnt/shared_data/finflow/gfw_clean/"

print("Spinning up FRESH GFW Production Worker...")
processor = GFWProductionWorker.options(name="GFW_Prod_Worker", lifetime="detached").remote()

print("Submitting massive GFW processing job to cluster...")
result = ray.get(processor.process_all.remote(input_pattern, output_path))

print(result)

#### Reconnect to detached Actor 

In [5]:
input_pattern = "/mnt/shared_data/finflow/gfw_raw/**/*.parquet"

output_path = "/mnt/shared_data/finflow/gfw_clean/"

processor = ray.get_actor("GFW_Prod_Worker")

print("Submitting GFW processing job to cluster...")
# We submit the job exactly the same way
result = ray.get(processor.process_all.remote(input_pattern, output_path))

print(result)

Submitting massive GFW processing job to cluster...


(raylet) Spilled 5558 MiB, 98 objects, write throughput 637 MiB/s.


Success! All 168 GFW files processed and saved to /mnt/shared_data/finflow/gfw_clean/


#### Kill Detached Actor

In [7]:
# 1. Ensure your remote is connected
if not ray.is_initialized():
    ray.init(address="auto", namespace="GFW_PROD_CLEAN")

print("Hunting down the immortal worker...")
try:
    # 2. Find the worker by its permanent name
    worker = ray.get_actor("GFW_Prod_Worker")
    
    # 3. Terminate it immediately
    ray.kill(worker)
    print("Success! The worker has been terminated and the RAM is freed.")
    
except ValueError:
    print("The worker is already dead or couldn't be found.")

Hunting down the immortal worker...
Success! The worker has been terminated and the RAM is freed.


## Sanity Check

In [3]:
import glob
import duckdb

# Identify a single file from your clean GFW directory
gfw_files = glob.glob("/mnt/shared_data/finflow/gfw_clean/*.parquet")

if not gfw_files:
    print("No files found! Check your mount point.")
else:
    single_file = gfw_files[0]
    print(f"Inspecting file: {single_file}")

    # Run sanity stats
    query = f"""
        SELECT 
            COUNT(*) as row_count,
            MAX(fishing_hours) as max_hours,
            MIN(fishing_hours) as min_hours,
            AVG(fishing_hours) as avg_hours,
            COUNT(DISTINCT h3_index) as unique_hexagons
        FROM read_parquet('{single_file}')
    """
    
    sanity_df = duckdb.query(query).df()
    display(sanity_df)

Inspecting file: /mnt/shared_data/finflow/gfw_clean/6_dd5086d54a4242b5a00c558cdbab9ca0_000707_000000-0.parquet


,row_count,max_hours,min_hours,avg_hours,unique_hexagons
0,1048576,5019.0,1.0,2.222116,781203
